# Training Models

## 1. Import libraries

In [1]:
import pandas as pd
import torch
import optuna
import optuna.visualization as vis

from src.tuning.tune import tune
from src.data.class_to_idx import MULTICLASS_IDX, BINARY_IDX, FILTERED_IDX

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\mpadillaa\Desktop\EILO-BioAcoustics-Analysis\.venv\Lib\site-packages\torch\cuda\__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2. Load data

In [2]:
df = pd.read_csv("splits\\splits.csv")
df

,id,name,age,gender,position,record_id,segment,label,category,duration,fold
0,1,P1,4.3,1,p4,7545,0,Normal,Normal,1.57725,-1
1,2,P1,4.3,1,p4,7545,1,Rhonchi,Adventitious,0.95725,-1
2,3,P1,4.3,1,p4,7545,2,Normal,Normal,1.01225,-1
3,4,P2,5.3,0,p1,25271,0,Normal,Normal,2.12525,-1
4,9,P3,4.3,1,p1,24116,0,Normal,Normal,2.40825,5
...,...,...,...,...,...,...,...,...,...,...,...
20866,24574,P957,8.4,0,p8,32670,3,Normal,Normal,0.94025,3
20867,24575,P957,8.4,0,p8,32670,4,Normal,Normal,0.99525,3
20868,24576,P957,8.4,0,p8,32670,5,Normal,Normal,0.68525,3
20869,24577,P957,8.4,0,p8,32670,6,Normal,Normal,1.19925,3


## 3. Hyperparameter tuning

In [3]:
df_config = {
    "image_col": "id",
    "label_col": "label",
    "fold_col": "fold",
}

origin = "..\\dataset\\dataset\\spectrogram"

class_to_idx = FILTERED_IDX

study_name = "STFT"

In [4]:
study = tune(
    df,
    df_config,
    origin,
    class_to_idx,
    study_name,
    fold=1,
    n_trials=5,
    in_channels=1,
    num_classes=4,
    save=True
)

[I 2026-10-07 16:51:13,467] A new study created in RDB with name: STFT


  0%|          | 0/5 [00:00<?, ?it/s]

Caching images:   0%|          | 0/13358 [00:00<?, ?it/s]

Caching images:   0%|          | 0/3339 [00:00<?, ?it/s]

Training:   0%|          | 0/120 [00:00<?, ?epoch/s]

c:\Users\mpadillaa\Desktop\EILO-BioAcoustics-Analysis\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Train]:   0%|          | 0/417 [00:19<?, ?batch/s]

[W 2026-10-07 16:51:59,389] Trial 0 failed with parameters: {'depth': 3, 'base_filters': 8, 'alpha_leaky_relu': 0.0024348773534554596, 'embedding_dim': 128, 'hidden_dim': 32, 'dropout_cnn': 0.052475643163223784, 'dropout_fc': 0.17277800745684632, 'lr_max': 0.00011963764382790327, 'weight_decay': 0.0034239600477873906, 'gamma_focal': 0.5579754426081673} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\mpadillaa\Desktop\EILO-BioAcoustics-Analysis\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\mpadillaa\Desktop\EILO-BioAcoustics-Analysis\classification\src\tuning\tune.py", line 293, in <lambda>
    lambda trial: objective(
                  ~~~~~~~~~^
        trial=trial,
        ^^^^^^^^^^^^
    ...<8 lines>...
        device=device,
        ^^^^^^^^^^^^^^
    ),
    ^
  File "c:\Users\mpadillaa\Desktop\EILO-BioAcoustics-Analysis\classification\src\tunin

KeyboardInterrupt: 

In [5]:
study = optuna.load_study(study_name="STFT", storage=r"sqlite:///classification\results\optuna\STFT\STFT.db")

In [ ]:
# General view
vis.plot_optimization_history(study).show()

# Param importances
vis.plot_param_importances(study).show()

# Pruning verification
vis.plot_intermediate_values(study).show()

# Best hyperparameters
top_params = list(optuna.importance.get_param_importances(study).keys())[:5]
vis.plot_parallel_coordinate(study, params=top_params).show()
vis.plot_slice(study, params=top_params).show()

In [ ]:
# from pathlib import Path


# def save_figure(fig, output_dir, filename, formats=("png", "svg", "html"), scale=2):
#     """
#     Guarda una figura de Plotly (la que devuelven las funciones de
#     optuna.visualization) en distintos formatos.

#     Args:
#         fig: figura de plotly (go.Figure) devuelta por optuna.visualization.plot_*
#         output_dir: carpeta donde guardar
#         filename: nombre base del archivo, sin extensión
#         formats: cualquier combinación de "png", "svg", "html"
#         scale: factor de resolución para PNG (2 = doble resolución, útil para memorias/informes)
#     """
#     output_dir = Path(output_dir)
#     output_dir.mkdir(parents=True, exist_ok=True)

#     if "png" in formats:
#         fig.write_image(output_dir / f"{filename}.png", scale=scale)

#     if "svg" in formats:
#         fig.write_image(output_dir / f"{filename}.svg")

#     if "html" in formats:
#         # Versión interactiva: zoom, hover con valores exactos, pan, etc.
#         fig.write_html(output_dir / f"{filename}.html", include_plotlyjs="cdn")

In [ ]:
# import optuna.visualization as vis

# FIGURES_PATH = RESULTS_OPTUNA_PATH / study_name / "figures"

# fig1 = vis.plot_optimization_history(study)
# save_figure(fig1, FIGURES_PATH, "optimization_history")

# fig2 = vis.plot_param_importances(study)
# save_figure(fig2, FIGURES_PATH, "param_importances")

# fig3 = vis.plot_intermediate_values(study)
# save_figure(fig3, FIGURES_PATH, "intermediate_values")

# top_params = list(optuna.importance.get_param_importances(study).keys())[:5]

# fig4 = vis.plot_parallel_coordinate(study, params=top_params)
# save_figure(fig4, FIGURES_PATH, "parallel_coordinate")

# fig5 = vis.plot_slice(study, params=top_params)
# save_figure(fig5, FIGURES_PATH, "slice")

In [6]:
for trial in study.trials:
    print(f"Trial {trial.number}")
    print(f"  State: {trial.state}")
    print(f"  Value: {trial.value}")
    print(f"  Params: {trial.params}")
    print()

Trial 0
  State: TrialState.FAIL
  Value: None
  Params: {'depth': 3, 'base_filters': 8, 'alpha_leaky_relu': 0.0024348773534554596, 'embedding_dim': 128, 'hidden_dim': 32, 'dropout_cnn': 0.052475643163223784, 'dropout_fc': 0.17277800745684632, 'lr_max': 0.00011963764382790327, 'weight_decay': 0.0034239600477873906, 'gamma_focal': 0.5579754426081673}

